In [1]:
# input
fasta_file = "./tmp/sampled.fasta"
pred_file = "./data/sampled_pred.tsv"
# output
output_json_file = "./tmp/protenix_input.json"

In [2]:
metal_type_str_to_input = {
"0": ("ion", "ZN"),
"1": ("ion", "CA"),
"2": ("ion", "MG"),
"3": ("ion", "MN"),
"4": ("ion", "FE"),
"5": ("ion", "CU"),
"6": ("ion", "NI"),
"7": ("ion", "CO"),
"8": ("ligand", "CCD_SF4"),
"9": ("ligand", "CCD_FES"),
"10": ("ligand", "CCD_F3S"),
}

In [3]:
from itertools import combinations
import networkx as nx

def get_site(site_str: str):
    g = nx.Graph()
    sites = site_str.split(";")
    for s in sites:
        members = s.split(",")
        for (i, j) in combinations(members, 2):
            g.add_edge(i, j)
    
    result = []
    for s in nx.connected_components(g):
        result.append(sorted(list(s)))
    return result

In [4]:
import pandas as pd
from Bio import SeqIO

id2seq = dict()
for r in SeqIO.parse(fasta_file, "fasta"):
    id2seq[r.id] = str(r.seq)


df = pd.read_table(pred_file)
records = []
for _, row in df.iterrows():
    seq = id2seq[f"AFDB:AF-{row['seq_id']}-F1"]

    posi2metal = dict(zip(
        [str(int(i) - 1) for i in row['pred_seq_num'].split(",")],
        row['metal_type'].split(",")
    ))
    sites = get_site(row["site"])

    metal2num = dict()
    for site in sites:
        metals = set([posi2metal[i] for i in site])
        for m in metals:
            if m in metal2num.keys():
                metal2num[m] += 1
            else:
                metal2num[m] = 1

    metal_dict_list = []
    for m, num in metal2num.items():
        ligand_type, metal_str = metal_type_str_to_input[m]
        if ligand_type == "ion":
            metal_dict_list.append({
                "ion": {
                    "ion": metal_str,
                    "count": num
                }
            })
        elif ligand_type == "ligand":
            metal_dict_list.append({
                "ligand": {
                    "ligand": f"{metal_str}",
                    "count": num
                }
            })
        else:
            raise ValueError

    records.append({
        "name": row['seq_id'],
        "sequences": [
            {
                "proteinChain": {
                    "sequence": seq,
                    "count": 1,
                    "msa": {
                        "precomputed_msa_dir": f"./tmp/msa_protenix/{row['seq_id']}/0",
                        "pairing_db": "uniref100",
                    }
                },
                
            },
        ] + metal_dict_list
    })

In [5]:
import json

with open(output_json_file, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=4)